In [1]:
!pip install -q sentence-transformers faiss-cpu langchain-text-splitters

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Everything loaded successfully")

Everything loaded successfully


In [3]:
text = """

Machine Learning is a branch of Artificial Intelligence that allows systems
to learn from data without being explicitly programmed.

Supervised Learning uses labeled data.

Examples include classification and regression.

Unsupervised Learning uses unlabeled data.

Examples include clustering.

Reinforcement Learning learns through interaction.

Linear Regression predicts continuous values.

Logistic Regression is used for classification.

Decision Trees create decision rules.

Random Forest combines multiple trees.

Deep Learning uses neural networks.

Natural Language Processing helps computers understand language.

Large Language Models are trained on massive datasets.

RAG stands for Retrieval Augmented Generation.

RAG combines retrieval and text generation.

Embeddings convert text into vectors.

Vector databases store embeddings.

FAISS performs similarity search.

Chunking divides large text into smaller pieces.

"""

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30
)

chunks = splitter.split_text(text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print("\nChunk", i)
    print(chunk)

Number of chunks: 8

Chunk 0
Machine Learning is a branch of Artificial Intelligence that allows systems
to learn from data without being explicitly programmed.

Chunk 1
Supervised Learning uses labeled data.

Examples include classification and regression.

Unsupervised Learning uses unlabeled data.

Chunk 2
Examples include clustering.

Reinforcement Learning learns through interaction.

Linear Regression predicts continuous values.

Chunk 3
Logistic Regression is used for classification.

Decision Trees create decision rules.

Random Forest combines multiple trees.

Chunk 4
Deep Learning uses neural networks.

Natural Language Processing helps computers understand language.

Chunk 5
Large Language Models are trained on massive datasets.

RAG stands for Retrieval Augmented Generation.

RAG combines retrieval and text generation.

Chunk 6
Embeddings convert text into vectors.

Vector databases store embeddings.

FAISS performs similarity search.

Chunk 7
Chunking divides large text in

In [5]:
embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [6]:
chunk_embeddings = embedding_model.encode(chunks)

print("Embedding shape:")
print(chunk_embeddings.shape)

Embedding shape:
(8, 384)


In [7]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(chunk_embeddings)
)

print("Embeddings stored in FAISS")

Embeddings stored in FAISS


In [8]:
query = "What is RAG?"

print("User Question:")
print(query)

User Question:
What is RAG?


In [9]:
query_embedding = embedding_model.encode(
    [query]
)

print(query_embedding.shape)

(1, 384)


In [10]:
distance, indices = index.search(
    np.array(query_embedding),
    k=2
)

print(indices)

[[5 7]]


In [11]:
retrieved_chunks = []

for i in indices[0]:
    retrieved_chunks.append(
        chunks[i]
    )

context = " ".join(retrieved_chunks)

print(context)

Large Language Models are trained on massive datasets.

RAG stands for Retrieval Augmented Generation.

RAG combines retrieval and text generation. Chunking divides large text into smaller pieces.


In [12]:
!pip install -q transformers accelerate sentencepiece

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np

In [14]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM Loaded")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM Loaded


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("FLAN model loaded")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


FLAN model loaded


In [16]:
embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [17]:
chunk_embeddings = embedding_model.encode(chunks)

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(chunk_embeddings)
)

print("FAISS recreated")

FAISS recreated


In [22]:
query = input("Ask your question: ")

query_embedding = embedding_model.encode(
    [query]
)

distance, indices = index.search(
    np.array(query_embedding),
    k=2
)

print(indices)

Ask your question: What are embeddings?
[[6 4]]


In [25]:
retrieved_chunks = []

for i in indices[0]:
    retrieved_chunks.append(
        chunks[i]
    )

context = " ".join(retrieved_chunks)

print(context)

Embeddings convert text into vectors.

Vector databases store embeddings.

FAISS performs similarity search. Deep Learning uses neural networks.

Natural Language Processing helps computers understand language.


In [26]:
prompt = f"""
Answer only from provided context.

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


Answer only from provided context.

Context:
Embeddings convert text into vectors.

Vector databases store embeddings.

FAISS performs similarity search. Deep Learning uses neural networks.

Natural Language Processing helps computers understand language.

Question:
What are embeddings?

Answer:



In [27]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

convert text into vectors


In [28]:
def ask_rag(question):

    query_embedding = embedding_model.encode(
        [question]
    )

    distance, indices = index.search(
        np.array(query_embedding),
        k=2
    )

    retrieved_chunks=[]

    for i in indices[0]:
        retrieved_chunks.append(
            chunks[i]
        )

    context=" ".join(retrieved_chunks)

    prompt=f"""
    Answer only from the provided context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=50
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [29]:
print(
    ask_rag(
        "What is Deep Learning?"
    )
)

Machine Learning


In [30]:
print(
    ask_rag(
        "What are embeddings?"
    )
)

convert text into vectors


In [32]:
while True:

    question = input("You: ")

    if question.lower()=="exit":
        break

    answer = ask_rag(question)

    print("Bot:",answer)

You:  What are embeddings?
Bot: convert text into vectors
You: exit


In [33]:
!pip install -q gradio

In [ ]:
import gradio as gr

def chat(message, history):

    response = ask_rag(message)

    return response


demo = gr.ChatInterface(
    fn=chat,
    title="Traditional RAG Chatbot",
    description="Ask questions from your knowledge base"
)

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c6bbf7852ae0e59958.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
import json

with open("/content/Traditional_RAG_Project.ipynb", "r") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open("/content/Traditional_RAG_Project_clean.ipynb", "w") as f:
    json.dump(nb, f)

print("Clean notebook created")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Traditional_RAG_Project.ipynb'

In [2]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']
